<a href="https://colab.research.google.com/github/DarshiniMahesh/Chatbots-AI/blob/main/Optimized.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              🤖 AI RESUME OPTIMIZER — Mistral-7B            ║
# ║        Runtime → Change runtime type → T4 GPU first!        ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Install ──────────────────────────────────────────────────────
import subprocess
subprocess.run(["pip","install","transformers","torch","accelerate",
                "bitsandbytes","sentencepiece","gradio","PyPDF2","-q"], check=True)

# ── Imports ───────────────────────────────────────────────────────
import torch, io
import gradio as gr
import PyPDF2
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata


# ── Load Model ────────────────────────────────────────────────────
print("⏳ Loading Mistral-7B... (3–5 min, please wait)")
login(token=userdata.get("Token"))
MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model     = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token               = HF_TOKEN,
    quantization_config = BitsAndBytesConfig(
        load_in_4bit           = True,
        bnb_4bit_quant_type    = "nf4",
        bnb_4bit_compute_dtype = torch.float16
    ),
    device_map = "auto"
)
print("✅ Model ready!")

# ── Optimize Function ─────────────────────────────────────────────
def optimize_resume(resume_text):
    if not resume_text.strip():
        return "⚠️ Please provide your resume."

    prompt = f"""You are a professional resume writer with 10+ years of experience.

Optimize the resume below. Make it professional, impactful, and ATS-friendly.

RULES:
1. Keep all section headings exactly as in the original.
2. Keep all real facts unchanged — name, email, phone, college, CGPA, company names, project names.
3. Rewrite OBJECTIVE to be sharp, confident, and professional.
4. Keep all original SKILLS and add relevant professional ones that fit naturally.
5. Keep each project title exactly. Rewrite descriptions with strong action verbs and impact.
6. Rewrite INTERNSHIP duties with strong action verbs.
7. Make ACHIEVEMENTS crisp and impactful with bullet points.
8. Rewrite STRENGTHS to sound professional for a technical role.
9. Keep EDUCATION and HOBBIES exactly word-for-word. Do not change them at all.
10. Do NOT invent fake experience, skills, or certifications.
11. Output ONLY the optimized resume. No explanation. No extra text.

ORIGINAL RESUME:
{resume_text}

OPTIMIZED RESUME:"""

    inputs = tokenizer(
        f"[INST] {prompt.strip()} [/INST]",
        return_tensors = "pt",
        truncation     = True,
        max_length     = 3000
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens     = 1400,
            do_sample          = False,
            repetition_penalty = 1.15,
            pad_token_id       = tokenizer.eos_token_id
        )

    result = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens = True
    ).strip()

    for tag in ["OPTIMIZED RESUME:", "REWRITTEN RESUME:", "[INST]", "[/INST]", "Note:", "Here is", "Here's"]:
        result = result.replace(tag, "").strip()

    return result

# ── PDF Reader ────────────────────────────────────────────────────
def read_pdf(pdf_file):
    if pdf_file is None:
        return ""
    reader = PyPDF2.PdfReader(pdf_file.name)
    return "\n".join(p.extract_text() or "" for p in reader.pages).strip()

def run(resume_text, pdf_file):
    text = read_pdf(pdf_file) if pdf_file else resume_text.strip()
    if not text:
        return "⚠️ Please paste your resume or upload a PDF."
    return optimize_resume(text)

# ── Gradio UI ─────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="AI Resume Optimizer") as app:

    gr.Markdown("""
    # 🤖 AI Resume Optimizer
    **Paste or upload your resume — get a professional, job-ready version instantly.**
    """)

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📋 Your Resume")
            resume_input = gr.Textbox(
                placeholder = "Paste your resume text here...",
                label       = "Paste Resume Text",
                lines       = 20
            )
            pdf_input = gr.File(
                label      = "Or Upload Resume PDF",
                file_types = [".pdf"]
            )
            btn = gr.Button("✨ Optimize My Resume", variant="primary", size="lg")

        with gr.Column(scale=1):
            gr.Markdown("### ✅ Optimized Resume")
            output = gr.Textbox(
                label            = "Your Optimized Resume",
                lines            = 28,
                show_copy_button = True
            )

    btn.click(fn=run, inputs=[resume_input, pdf_input], outputs=output)

app.launch(share=True)